# 🏆 Kaggle House Prices: Advanced Regression Techniques
**Top 1% Target Pipeline — Ordinal Neighborhood Encoding & Dense Quality-SF Interactions**


In [ ]:
# =============================================================================
# 1. ENVIRONMENT & DEPENDENCY INITIALIZATION
# =============================================================================
import os, sys, gc, time, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize
from scipy.stats import skew
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from catboost import CatBoostRegressor

print("✅ Core Libraries Loaded Successfully!")


In [ ]:
# =============================================================================
# 2. TOP 1% SMALL-DATASET PREPROCESSING ENGINE
# =============================================================================
QUALITY_MAP = {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0, "No Garage": 0, "No Basement": 0}
BSMT_FIN_MAP = {"GLQ": 6, "ALQ": 5, "BLQ": 4, "Rec": 3, "LwQ": 2, "Unf": 1, "None": 0, "No Basement": 0}
EXPOSURE_MAP = {"Gd": 4, "Av": 3, "Mn": 2, "No": 1, "None": 0, "No Basement": 0}

kaggle_input = Path("/kaggle/input")
train_matches = list(kaggle_input.rglob("train.csv")) if kaggle_input.exists() else []
test_matches = list(kaggle_input.rglob("test.csv")) if kaggle_input.exists() else []

if train_matches and test_matches:
    train_path, test_path = train_matches[0], test_matches[0]
else:
    zip_path = Path("./data/house-prices-advanced-regression-techniques.zip")
    with zipfile.ZipFile(zip_path, "r") as z: z.extractall("./data_temp")
    train_path, test_path = Path("./data_temp/train.csv"), Path("./data_temp/test.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

train = train[~((train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000))].reset_index(drop=True)
y_train_log = np.log1p(train["SalePrice"].values)

# Neighborhood Ordinal Target Rank Mapping
neigh_order = train.groupby("Neighborhood")["SalePrice"].median().sort_values().index
neigh_map = {n: i + 1 for i, n in enumerate(neigh_order)}

X_train = train.drop(columns=["Id", "SalePrice"])
X_test = test.drop(columns=["Id"])
combined = pd.concat([X_train, X_test], ignore_index=True)
combined["Neighborhood"] = combined["Neighborhood"].map(neigh_map).fillna(13).astype(int)

ord_cols = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC", "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]
for col in ord_cols: combined[col] = combined[col].fillna("None").map(QUALITY_MAP).fillna(0).astype(int)
combined["BsmtFinType1"] = combined["BsmtFinType1"].fillna("None").map(BSMT_FIN_MAP).fillna(0).astype(int)
combined["BsmtFinType2"] = combined["BsmtFinType2"].fillna("None").map(BSMT_FIN_MAP).fillna(0).astype(int)
combined["BsmtExposure"] = combined["BsmtExposure"].fillna("None").map(EXPOSURE_MAP).fillna(0).astype(int)

cat_cols = [c for c in combined.select_dtypes(include=["object"]).columns]
for c in cat_cols: combined[c] = combined[c].fillna("None")
num_cols = [c for c in combined.select_dtypes(include=[np.number]).columns]
for c in num_cols: combined[c] = combined[c].fillna(combined[c].median())

combined["TotalSF"] = combined["TotalBsmtSF"] + combined["1stFlrSF"] + combined["2ndFlrSF"]
combined["TotalBath"] = combined["FullBath"] + (0.5 * combined["HalfBath"]) + combined["BsmtFullBath"] + (0.5 * combined["BsmtHalfBath"])
combined["TotalPorch"] = combined["OpenPorchSF"] + combined["3SsnPorch"] + combined["EnclosedPorch"] + combined["ScreenPorch"] + combined["WoodDeckSF"]
combined["Quality_SF_Score"] = combined["OverallQual"] * combined["TotalSF"]
combined["House_Age"] = (combined["YrSold"] - combined["YearBuilt"]).clip(lower=0)
combined["Remod_Age"] = (combined["YrSold"] - combined["YearRemodAdd"]).clip(lower=0)
combined["Is_New_House"] = (combined["YearBuilt"] == combined["YrSold"]).astype(int)

num_features = [c for c in combined.select_dtypes(include=[np.number]).columns]
skewed = combined[num_features].apply(lambda x: skew(x)).sort_values(ascending=False)
high_skew = skewed[abs(skewed) > 0.75].index
for c in high_skew: combined[c] = np.log1p(combined[c])

encoded = pd.get_dummies(combined, drop_first=True)
X_train_proc = encoded.iloc[:len(train)].copy()
X_test_proc = encoded.iloc[len(train):].copy()
print(f"✨ Top 1% Processed: Train={X_train_proc.shape}, Test={X_test_proc.shape}")


In [ ]:
# =============================================================================
# 3. HIGH-REGULARIZATION REGRESSION BLENDING (LassoCV + RidgeCV + ElasticNetCV + CatBoost)
# =============================================================================
kf = KFold(n_splits=10, shuffle=True, random_state=42)

lasso = LassoCV(alphas=np.logspace(-4, 1, 60), cv=5, max_iter=5000, random_state=42)
ridge = RidgeCV(alphas=np.logspace(-2, 3, 60), cv=5)
elastic = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9], cv=5, max_iter=5000, random_state=42)
cat = CatBoostRegressor(iterations=700, learning_rate=0.03, depth=4, l2_leaf_reg=5.0, verbose=0, random_seed=42)

oof_lasso = np.zeros(len(X_train_proc))
oof_ridge = np.zeros(len(X_train_proc))
oof_elastic = np.zeros(len(X_train_proc))
oof_cat = np.zeros(len(X_train_proc))

preds_lasso = np.zeros(len(X_test_proc))
preds_ridge = np.zeros(len(X_test_proc))
preds_elastic = np.zeros(len(X_test_proc))
preds_cat = np.zeros(len(X_test_proc))

for tr_idx, va_idx in kf.split(X_train_proc):
    X_tr, y_tr = X_train_proc.iloc[tr_idx], y_train_log[tr_idx]
    X_va, y_va = X_train_proc.iloc[va_idx], y_train_log[va_idx]
    
    lasso.fit(X_tr, y_tr)
    oof_lasso[va_idx] = lasso.predict(X_va)
    preds_lasso += lasso.predict(X_test_proc) / 10.0
    
    ridge.fit(X_tr, y_tr)
    oof_ridge[va_idx] = ridge.predict(X_va)
    preds_ridge += ridge.predict(X_test_proc) / 10.0
    
    elastic.fit(X_tr, y_tr)
    oof_elastic[va_idx] = elastic.predict(X_va)
    preds_elastic += elastic.predict(X_test_proc) / 10.0
    
    cat.fit(X_tr, y_tr)
    oof_cat[va_idx] = cat.predict(X_va)
    preds_cat += cat.predict(X_test_proc) / 10.0

oof_matrix = np.column_stack([oof_lasso, oof_ridge, oof_elastic, oof_cat])
test_matrix = np.column_stack([preds_lasso, preds_ridge, preds_elastic, preds_cat])

def rmse_objective(weights):
    blend = np.dot(oof_matrix, weights)
    return np.sqrt(mean_squared_error(y_train_log, blend))

init_weights = np.array([0.40, 0.25, 0.25, 0.10])
bounds = [(0, 1) for _ in range(4)]
constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1.0}
res = minimize(rmse_objective, init_weights, method="SLSQP", bounds=bounds, constraints=constraints)

best_w = res.x
print(f"🏆 Top 1% Target 10-Fold OOF RMSLE: {rmse_objective(best_w):.5f}")
print(f"⚖️ Optimal Weights: Lasso={best_w[0]:.3f}, Ridge={best_w[1]:.3f}, Elastic={best_w[2]:.3f}, CatBoost={best_w[3]:.3f}")


In [ ]:
# =============================================================================
# 4. SUBMISSION GENERATION & BOUNDARY CLIPPING
# =============================================================================
final_log_preds = np.dot(test_matrix, best_w)
final_price_preds = np.expm1(final_log_preds)
final_calibrated = np.clip(final_price_preds, 42000.0, 525000.0)

sub = pd.DataFrame({"Id": test["Id"].values, "SalePrice": final_calibrated})
sub.to_csv("submission.csv", index=False)
print(f"📄 Generated submission.csv for {len(sub)} predictions!")
print(sub.head())
